# 動能持股公告個案研究
目的：以歷年十月最後一個星期二為起點，按原 A 策略選股，再查核當時可知及其後 24 日的新公告。
沿用現有 aicup kernel；不重啟、不清除既有物件。新增探索結果留於 kernel。
此 Notebook 依賴既有動能研究狀態，尚非獨立重跑版本；沒有執行 Restart and Run All。

In [ ]:
# Locate this competition repository; do not restart an existing kernel.
import sys as _repo_sys, importlib.util as _repo_importlib
from pathlib import Path as _PlainPath
_candidates = [_PlainPath.cwd(), *_PlainPath.cwd().parents]
_repo_root = next((p for p in _candidates if (p / '99_維護與備份' / '工具' / '研究路徑.py').exists()), None)
if _repo_root is None:
    raise RuntimeError('Open Jupyter from this repository root or its 04_Notebook directory.')
if 'research_layout' not in _repo_sys.modules:
    _spec = _repo_importlib.spec_from_file_location('research_layout', _repo_root / '99_維護與備份' / '工具' / '研究路徑.py')
    _mod = _repo_importlib.module_from_spec(_spec)
    _repo_sys.modules['research_layout'] = _mod
    _spec.loader.exec_module(_mod)
from research_layout import ResearchPath as Path
%whos dict
import sys
print('Python:',sys.executable)
print('Required state:', {k: k in globals() for k in ['raw_data','stock_ids','dates','scores','close','adj_close','bad_action']})
print('Relevant names:', [k for k in globals() if k in ['dates','trade_dates','stock_names','name_map','root','shape','ret20','ret60','liquid','eligible','valid_prices']])

## 同一策略、同一時間規則選股
A＝20／60 日調整報酬的截面百分位平均。沿用既有研究 eligibility（包含法人／成交量完整性條件），不是另行更改選股器。使用目前官方 150 檔名單回看，有存活者／名單偏誤。公司行動篩查仍有資料品質限制。

In [ ]:
import calendar, datetime, json
historical_momentum_cases = []
historical_momentum_skipped = []
historical_momentum_years = [2023, 2024, 2025]
historical_momentum_top_n = 3
print('Stock metadata sample:', raw_data['stocks'][:1])
for case_year in historical_momentum_years:
    case_anchor = max(datetime.date(case_year,10,d) for d in range(1,32) if datetime.date(case_year,10,d).weekday()==1)
    if case_anchor not in di:
        historical_momentum_skipped.append({'year':case_year,'anchor':str(case_anchor),'reason':'anchor unavailable'})
        continue
    case_t = di[case_anchor]-1
    case_idx = np.flatnonzero(np.isfinite(scores['A'][case_t]))
    if not len(case_idx):
        historical_momentum_skipped.append({'year':case_year,'anchor':str(case_anchor),'previous_sessions':case_t+1,'reason':'60-day momentum warmup unavailable'})
        continue
    case_idx = sorted(case_idx,key=lambda j:(-float(scores['A'][case_t,j]),stock_ids[j]))
    for case_rank,case_j in enumerate(case_idx[:historical_momentum_top_n],1):
        case_rr = signal_returns[case_t-59:case_t+1,case_j]
        historical_momentum_cases.append({'year':case_year,'anchor':str(case_anchor),'signal_date':str(dates[case_t]),'window_end':str(dates[di[case_anchor]+23]),'rank':case_rank,'stock_id':stock_ids[case_j],'score':round(float(scores['A'][case_t,case_j]),6),'return20_pct':round(float(np.prod(1+case_rr[-20:])-1)*100,2),'return60_pct':round(float(np.prod(1+case_rr)-1)*100,2),'eligible_stocks':len(case_idx)})
print(json.dumps({'cases':historical_momentum_cases,'skipped':historical_momentum_skipped},ensure_ascii=False,indent=2))


## 唯讀取得入選股報告索引
公告原文與大型數值表留在 kernel；只輸出需要的摘要。DB 營收／財報缺原始公開時間及修訂版本，數值不可直接當成當時已知。公告 event_date 不等於 announce_date。

In [ ]:
from research_layout import ResearchPath as Path
import re, pymysql
def fetch_historical_case_documents(cases):
    cfg_text = next((Path.home()/'Downloads').glob('AI_CUP_DB*.md')).read_text(encoding='utf-8-sig')
    cfg = {k:v.strip().strip('`').strip('"').strip("'") for k,v in re.findall(r'(?m)^(DB_[A-Z_]+)\s*=\s*(.*)$',cfg_text)}
    del cfg_text
    dbc = None
    result = {}
    try:
        dbc = pymysql.connect(host=cfg['DB_HOST'],port=int(cfg['DB_PORT']),user=cfg['DB_USER'],password=cfg['DB_PASSWORD'],database=cfg['DB_NAME'],charset='utf8mb4',connect_timeout=15,read_timeout=90,autocommit=False,cursorclass=pymysql.cursors.DictCursor)
        cfg.clear()
        with dbc.cursor() as q:
            q.execute('SET SESSION MAX_EXECUTION_TIME=30000')
            q.execute('SET SESSION TRANSACTION READ ONLY')
            q.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
            for c in cases:
                sid,y = c['stock_id'],c['year']
                q.execute('SELECT stock_id,name,industry_name FROM stocks WHERE stock_id=%s',(sid,))
                metadata = q.fetchone()
                q.execute('SELECT ann_id,announce_date,announce_time,subject,event_date,content,source FROM announcements WHERE stock_id=%s AND announce_date BETWEEN %s AND %s ORDER BY announce_date,announce_time',(sid,f'{y}-01-01',c['window_end']))
                anns = q.fetchall()
                q.execute('SELECT `year_month`,revenue,revenue_last_month,revenue_last_year,mom_pct,yoy_pct,note FROM monthly_revenue WHERE stock_id=%s AND `year_month` BETWEEN %s AND %s ORDER BY `year_month`',(sid,f'{y-1}-07',f'{y}-10'))
                revs = q.fetchall()
                q.execute('SELECT period,`statement`,item,value FROM financials WHERE stock_id=%s AND period BETWEEN %s AND %s ORDER BY period,`statement`,item',(sid,f'{y-1}Q2',f'{y}Q3'))
                fins = q.fetchall()
                result[(y,sid)] = {'metadata':metadata,'announcements':anns,'revenues':revs,'financials':fins}
        return result
    except Exception as case_error:
        print('Safe query error:',type(case_error).__name__,case_error.args[0] if case_error.args and isinstance(case_error.args[0],int) else 'non-SQL error')
        raise RuntimeError('Read-only case query failed; connection details suppressed') from None
    finally:
        cfg.clear()
        if dbc is not None:
            dbc.rollback()
            dbc.close()
historical_case_documents = fetch_historical_case_documents(historical_momentum_cases)
for c in historical_momentum_cases:
    d = historical_case_documents[(c['year'],c['stock_id'])]
    c['name'] = d['metadata']['name'] if d['metadata'] else c['stock_id']
    relevant = [a for a in d['announcements'] if any(k in a['subject'] for k in ['法人','法說','財務報','財報','營收','營業額'])]
    before = [a for a in relevant if a['announce_date'] <= datetime.date.fromisoformat(c['signal_date'])]
    after = [a for a in relevant if a['announce_date'] >= datetime.date.fromisoformat(c['anchor'])]
    print(json.dumps({'year':c['year'],'stock':c['stock_id']+' '+c['name'],'counts':{k:len(d[k]) for k in ['announcements','revenues','financials']},'latest_revenue_rows':d['revenues'][-2:],'before_subjects':[{k:a[k] for k in ['announce_date','announce_time','subject','event_date']} for a in before[-4:]],'during_subjects':[{k:a[k] for k in ['announce_date','announce_time','subject','event_date']} for a in after],'financial_items':sorted({r['item'] for r in d['financials']})[:35]},ensure_ascii=False,default=str))


## 營收比較與資料完整性
M2M/MoM 為月增率，Y2Y/YoY 為年增率。由原始營收重算，另看近三月合計 YoY。十月數字屬起點之後的新資料；DB 未保存營收原始發布時間與修訂版本，不能直接用來做無前視偏誤回測。

In [ ]:
historical_case_revenue_summary = []
for case in historical_momentum_cases:
    y, sid = case['year'], case['stock_id']
    docs = historical_case_documents[(y,sid)]
    lookup = {str(r['year_month'])[:7]:r for r in docs['revenues']}
    for month in (9,10):
        row = lookup.get(f'{y}-{month:02d}')
        if not row: continue
        rev = float(row['revenue'])
        mom = 100*(rev/float(row['revenue_last_month'])-1) if row['revenue_last_month'] else None
        yoy = 100*(rev/float(row['revenue_last_year'])-1) if row['revenue_last_year'] else None
        recent = [lookup.get(f'{y}-{m:02d}') for m in range(month-2,month+1)]
        prior = [lookup.get(f'{y-1}-{m:02d}') for m in range(month-2,month+1)]
        yoy3 = None
        if all(recent) and all(r['revenue_last_year'] for r in recent):
            numerator=sum(float(r['revenue']) for r in recent)
            denominator=sum(float(r['revenue_last_year']) for r in recent)
            yoy3 = 100*(numerator/denominator-1) if denominator else None
        summary={'year':y,'stock_id':sid,'month':month,'revenue':rev,'mom_pct':mom,'yoy_pct':yoy,'three_month_yoy_pct':yoy3}
        historical_case_revenue_summary.append(summary)
        print(y,sid,month,'MoM',round(mom,2),'YoY',round(yoy,2),'3M YoY',None if yoy3 is None else round(yoy3,2))


## 原始報告入口與後續閱讀
官方連結及各股事件時間線見同目錄「動能持股公告個案研究_2026-10-14.md」。報告索引已建立，尚未完成六股全文逐項核對、歷史市場預期與部位調整回測。大型公告、營收、財報物件留在 historical_case_documents；歷史候選留在 historical_momentum_cases；營收摘要留在 historical_case_revenue_summary。

## 2026-10-14 修正研究範圍：A 完整 25 檔
固定每年 25 檔，不比較 20／25／30。先查資料，不指定公告加減碼幅度。原本六股保留為個案，不代表完整組合。以下沿用現有 kernel、訊號與查詢函式，保留原本物件。

In [ ]:
%whos dict
import sys
print('Python environment:',sys.executable)
print('Required state:',{k:k in globals() for k in ['scores','signal_returns','dates','di','stock_ids','raw_data','historical_case_documents','fetch_historical_case_documents']})
assert all(k in globals() for k in ['scores','signal_returns','dates','di','stock_ids','raw_data','historical_case_documents','fetch_historical_case_documents'])


In [ ]:
import datetime, json
historical_a25_cases=[]
historical_a25_skipped=[]
for y in [2023,2024,2025]:
    anchor=max(datetime.date(y,10,d) for d in range(1,32) if datetime.date(y,10,d).weekday()==1)
    t=di[anchor]-1
    eligible=np.flatnonzero(np.isfinite(scores['A'][t]))
    if len(eligible)<25:
        historical_a25_skipped.append({'year':y,'anchor':str(anchor),'eligible':len(eligible),'reason':'Insufficient original A warmup/data'})
        continue
    order=sorted(eligible,key=lambda j:(-float(scores['A'][t,j]),stock_ids[j]))[:25]
    for rank,j in enumerate(order,1):
        rr=signal_returns[t-59:t+1,j]
        historical_a25_cases.append({'year':y,'anchor':str(anchor),'signal_date':str(dates[t]),'window_end':str(dates[di[anchor]+23]),'rank':rank,'stock_id':stock_ids[j],'score':float(scores['A'][t,j]),'return20_pct':float(np.prod(1+rr[-20:])-1)*100,'return60_pct':float(np.prod(1+rr)-1)*100,'target_weight_pct':3.6,'eligible_stocks':len(eligible)})
for y in [2024,2025]:
    cases=[c for c in historical_a25_cases if c['year']==y]
    assert len(cases)==25 and len({c['stock_id'] for c in cases})==25
    assert abs(sum(c['target_weight_pct'] for c in cases)-90)<1e-8
print('Selected per year:',{y:sum(c['year']==y for c in historical_a25_cases) for y in [2024,2025]},'Skipped:',historical_a25_skipped)
historical_a25_documents=dict(historical_case_documents)
missing=[c for c in historical_a25_cases if (c['year'],c['stock_id']) not in historical_a25_documents]
historical_a25_documents.update(fetch_historical_case_documents(missing))
for c in historical_a25_cases:
    meta=historical_a25_documents[(c['year'],c['stock_id'])]['metadata'] or {}
    c['name']=meta.get('name',c['stock_id'])
    c['industry']=meta.get('industry_name') or '未分類'
print('Reused case pairs:',50-len(missing),'New read-only case queries:',len(missing))
print('Selections:',[(c['year'],c['rank'],c['stock_id'],c['name']) for c in historical_a25_cases])


## 查閱報告輸出
目標 3.6% 是等權配置目標，不是已成交權重或建議加減碼。DB 數值是目前歷史快照，營收與財報缺原始發布時間／版本。報告列出公告日期時間與原文供核對；不以 period 或 event_date 代替首次公開时间。

In [ ]:
from research_layout import ResearchPath as Path
from collections import Counter
import html
historical_a25_revenue_summary=[]
def a25_revenue(c,month):
    rows=historical_a25_documents[(c['year'],c['stock_id'])]['revenues']
    lookup={str(r['year_month'])[:7]:r for r in rows}
    y=c['year']; r=lookup.get(f'{y}-{month:02d}')
    if not r:return None
    def growth(v,b):return 100*(float(v)/float(b)-1) if v is not None and b is not None and float(b)>0 else None
    recent=[lookup.get(f'{y}-{m:02d}') for m in range(month-2,month+1)]
    three=None
    if all(recent) and all(v['revenue'] is not None and v['revenue_last_year'] is not None for v in recent):
        three=growth(sum(float(v['revenue']) for v in recent),sum(float(v['revenue_last_year']) for v in recent))
    s={'year':y,'stock_id':c['stock_id'],'month':month,'revenue':r['revenue'],'mom':growth(r['revenue'],r['revenue_last_month']),'yoy':growth(r['revenue'],r['revenue_last_year']),'three_month_yoy':three,'note':r['note']}
    historical_a25_revenue_summary.append(s)
    return s
def a25_pct(v):return '缺資料' if v is None else f'{v:.2f}%'
def a25_escape(v):return str(v).replace('|','／').replace('\n',' ').replace('\r',' ')
def a25_stage(a,c):
    ad=str(a['announce_date'])[:10]
    if ad<c['anchor']:return '起點前'
    if ad==c['anchor']:
        tm=str(a.get('announce_time') or '')
        if len(tm)==8 and tm<'08:55:00':return '起點當日截止前（需再核對資料可得性）'
        return '區間新增／起點當日時點待核對'
    return '區間新增'
report=['# A 策略：歷史同季起點完整 25 檔查閱資料','',
'更新：2026-10-14。固定 A 每年 25 檔，先提供查閱資料，不做持股數比較或公告加減碼配置。原本每年前三檔只是個案；本表補上完整 25 檔。','',
'## 讀法與限制','',
'- 選股：20／60 日調整後報酬的百分位平均，沿用原 A 可選條件與目前 150 檔名單。排名只用來選股，不決定投入比例。法人資料完整性是原可選條件之一，但 A 沒有法人加分因子。',
'- 原 A 每五個交易日調整；本文件列的是各年起點的 25 檔，不是整段 24 日都固定不換股。若要固定持有，對應 AH 的研究設定。',
'- 每檔目標占總資產 3.6%，25 檔合計 90%，現金目標 10%。整張取整、成交價、費用及價格漂移會使實際權重不同；本輪沒有重建初始成交帳戶。',
'- 2024：10/29 起點、10/28 價格訊號、24 日至 12/2；2025：10/28 起點、10/27 價格訊號、24 日至 11/28。2023 缺 60 日暖機資料，排除。',
'- 營收及財報為目前 DB 歷史快照，缺首次公開時間／修訂版本。九月營收需核對發布時間後才能當成起點已知；十月數字不可提前使用。',
'- M2M／MoM＝月增率，Y2Y／YoY＝年增率。近三月 YoY 用合計營收與同列去年同期金額重算，不平均百分比。',
'- 財報 Q2／Q3 可能是累計而非單季；DB 單位及報表範圍尚待原件核對。以下原始數值不直接稱為單季 EPS，也不推導加減碼。',
'- 公告時間可供追溯，event_date 不代表首次公開時間；法說預告可能在起點前、實際會議在區間內。原文保留在各股明細。',
'- 本輪沒有完整爬取新聞、川普／馬斯克貼文，也沒有新增權重調整回測。原 A 回測扣買賣各 0.1425% 手續費與賣出稅 0.3%。','']
for y in [2024,2025]:
    cases=[c for c in historical_a25_cases if c['year']==y]
    report += [f'## {y} 年完整 25 檔','', '|排名|股票|DB 產業分類|20 日報酬|60 日報酬|A 分數|目標權重|9 月 MoM|9 月 YoY|7–9 月 YoY|10 月 MoM|10 月 YoY|8–10 月 YoY|','|---:|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
    for c in cases:
        s=a25_revenue(c,9);o=a25_revenue(c,10)
        vals=[c['rank'],c['stock_id']+' '+c['name'],c['industry'],a25_pct(c['return20_pct']),a25_pct(c['return60_pct']),f"{c['score']:.4f}",'3.6%']
        vals += [a25_pct(r.get(k) if r else None) for r in [s,o] for k in ['mom','yoy','three_month_yoy']]
        report.append('|'+ '|'.join(a25_escape(v) for v in vals)+'|')
    sectors=Counter(c['industry'] for c in cases)
    report += ['',f'### {y} 產業集中度（按等權目標計）','', '|DB 產業分類|檔數|占總資產目標|','|---|---:|---:|']
    report += [f'|{a25_escape(k)}|{v}|{v*3.6:.1f}%|' for k,v in sectors.most_common()]
    report += ['','DB 分類可能較粗，電子子產業的共同曝險尚未細分，不能由此宣稱已充分分散。','']
report += ['## 逐檔公告與財報查閱','',
'官方原件查詢入口：[公開資訊觀測站](https://mops.twse.com.tw/mops/#/web/home)。依股票代號、民國年度（2024＝113、2025＝114）、季度及公告日期查詢。這是查詢入口，不是假稱已定位的原始 PDF。已找到的六股官方原件見同目錄 [個案研究](動能持股公告個案研究_2026-10-14.md)。','']
coverage=[]
for c in historical_a25_cases:
    docs=historical_a25_documents[(c['year'],c['stock_id'])]
    relevant=[a for a in docs['announcements'] if any(k in (a.get('subject') or '') for k in ['法人','法說','財務報','財報','營收','營業額'])]
    before=[a for a in relevant if str(a['announce_date'])[:10]<c['anchor']][-5:]
    during=[a for a in relevant if str(a['announce_date'])[:10]>=c['anchor']]
    show=before+during
    coverage.append({'year':c['year'],'stock_id':c['stock_id'],'revenue_rows':len(docs['revenues']),'financial_rows':len(docs['financials']),'selected_announcements':len(show),'before_recent':len(before),'during':len(during)})
    report += [f"### {c['year']}／{c['rank']:02d}：{c['stock_id']} {c['name']}",'',f"目標權重 3.6%；動能排名 {c['rank']}。加碼／減碼：尚未判斷。",'', '|時段|發布日期／時間|公告編號|主題|公告 event_date|','|---|---|---|---|---|']
    for a in show:
        vals=[a25_stage(a,c),str(a['announce_date'])+' '+str(a.get('announce_time') or '時間缺漏'),a['ann_id'],a['subject'],a.get('event_date') or '—']
        report.append('|'+ '|'.join(a25_escape(v) for v in vals)+'|')
    if not show:report.append('|—|—|—|DB 未找到符合主題公告，不能推論沒有公告|—|')
    report += ['', '<details><summary>公告原文與來源欄位（展開查閱）</summary>','']
    for a in show:
        report += [f"#### {a['announce_date']} {a.get('announce_time') or ''}：{a25_escape(a['subject'])}",'',f"DB source 欄位：{a25_escape(a.get('source') or '缺漏')}；公告編號：{a['ann_id']}。",'', '<pre>'+html.escape(str(a.get('content') or '原文缺漏'))+'</pre>','']
    report += ['</details>','','<details><summary>Q2／Q3 財報原始欄位（單季／累計與單位待核對）</summary>','', '|period|statement|item|原始 value|','|---|---|---|---:|']
    fs=[r for r in docs['financials'] if str(r['period']) in [f"{c['year']}Q2",f"{c['year']}Q3"]]
    for r in fs:report.append('|'+ '|'.join(a25_escape(r[k]) for k in ['period','statement','item','value'])+'|')
    if not fs:report.append('|—|—|未找到該年 Q2／Q3|—|')
    report += ['','</details>','']
report += ['## 驗證與下一步','',
'已驗證：兩年各 25 檔且不重複；目標權重合計各 90%。兩年原前三名與原個案選股一致。取用已有六個案例資料，只唯讀補查其餘案例；未修改團隊 DB。',
'現有 kernel 保留，Notebook 已保存程式與摘要輸出；依賴原動能資料物件，未重啟後 Run All，尚非獨立重跑驗證。',
'待核對：營收首次發布時間與修訂版本、財報單位／期間、每份法說原件與實際會議時間、新聞／政策曝險、逐檔投資理由及配置判斷。公告主題搜尋可能漏掉其他名稱的事件，不是完整事件普查。',
'本文件不替使用者決定加減碼；應先查閱原件，再建立可重複判讀規則。','']
for y in [2024,2025]:
    assert [(c['stock_id'],c['rank']) for c in historical_a25_cases if c['year']==y and c['rank']<=3]==[(c['stock_id'],c['rank']) for c in historical_momentum_cases if c['year']==y]
historical_a25_coverage=coverage
historical_a25_report_path=Path.cwd()/'資料探勘與應用'/'B_Final Project (0920~1221)'/'競賽資料'/'A策略_歷史25檔持股與公告查閱_2026-10-14.md'
historical_a25_report_path.write_text('\n'.join(report),encoding='utf-8')
assert historical_a25_report_path.read_text(encoding='utf-8').count('目標權重 3.6%；動能排名')==50
print('Report:',historical_a25_report_path)
print('Report characters:',historical_a25_report_path.stat().st_size,'Cases:',len(coverage),'Revenue observations:',len(historical_a25_revenue_summary))
print('Missing revenue cases:',[(r['year'],r['stock_id']) for r in coverage if r['revenue_rows']==0])
print('Missing financial cases:',[(r['year'],r['stock_id']) for r in coverage if r['financial_rows']==0])
print('Industry counts:',{y:dict(Counter(c['industry'] for c in historical_a25_cases if c['year']==y)) for y in [2024,2025]})


## 第一批原件判讀：力旺 2024／南亞科 2025
原始官方報告的閱讀證據、頁碼與解釋追加於同目錄個案 Markdown。以下只核對數值與事件交易日，不讀起點後報酬來選判斷。正式 Q3 財報原 PDF 本輪未能解析，因此 DB 差額只用於核對，不能冒充原件驗證。

In [ ]:
%whos dict
import datetime, json
assert 'historical_a25_documents' in globals() and 'historical_a25_cases' in globals()
report_reading_numeric_checks=[]
for key in [(2024,'3529'),(2025,'2408')]:
    y,sid=key; d=historical_a25_documents[key]
    selected={}
    for period in [f'{y}Q2',f'{y}Q3']:
        selected[period]={r['item']:float(r['value']) for r in d['financials'] if str(r['period'])==period and r['statement']=='income' and r['item'] in ['營業收入','營業利益（損失）','本期淨利（淨損）','淨利（淨損）歸屬於母公司業主','基本每股盈餘（元）']}
    cumulative_diff={k:selected[f'{y}Q3'][k]-selected[f'{y}Q2'][k] for k in selected[f'{y}Q3'] if k in selected[f'{y}Q2'] and k!='基本每股盈餘（元）'}
    revlookup={str(r['year_month'])[:7]:r for r in d['revenues']}
    q3rev=sum(float(revlookup[f'{y}-{m:02d}']['revenue']) for m in [7,8,9])
    event_date=datetime.date(y,11,8) if sid=='3529' else datetime.date(y,11,4)
    case=next(c for c in historical_a25_cases if (c['year'],c['stock_id'])==key)
    candidates=[dt for dt in dates if dt>event_date and str(dt)<=case['window_end']]
    check={'year':y,'stock_id':sid,'q3_monthly_sum_ntd':q3rev,'DB_income_raw':selected,'DB_q3_minus_q2_excluding_EPS':cumulative_diff,'event_date':str(event_date),'next_session_after_event':str(candidates[0]),'sessions_remaining_from_next':len(candidates)}
    report_reading_numeric_checks.append(check)
    print(json.dumps(check,ensure_ascii=False))
em=historical_a25_documents[(2024,'3529')]
octrow=next(r for r in em['revenues'] if str(r['year_month'])[:7]=='2024-10')
print('eMemory October same-month check:',{k:str(octrow[k]) for k in ['revenue','revenue_last_month','revenue_last_year']})
print('Policy: no forward stock returns queried; no rounded cumulative EPS subtraction; no weights changed.')


### 判讀狀態
力旺：季度營收認列節奏已在官方簡報找到；Q3 原件閱讀不能當成 11/8 首次公開財務數字。南亞科：10/13 自結為起點已知，11/4 累計虧損不能直接當成 Q3 單季虧損。兩檔仍只形成「維持原 A 配置候選／不足以加碼」的研究判讀，未形成正式交易建議或驗證的配置規則。

## 第二批：鈊象與聖暉 2024
保留原 25 檔，不讀後續報酬；核對原公告、財報口徑與官方原件。工程案在手訂單不是已實現收入，子公司報告不直接當母公司合併結果。

In [ ]:
%whos dict
import json,datetime
second_batch_numeric_checks=[]
for sid in ['3293','5536']:
    d=historical_a25_documents[(2024,sid)]
    selected={}
    for period in ['2024Q1','2024Q2','2024Q3']:
        selected[period]={r['item']:float(r['value']) for r in d['financials'] if str(r['period'])==period and r['statement']=='income' and r['item'] in ['營業收入','營業毛利（毛損）','營業利益（損失）','本期淨利（淨損）','淨利（淨損）歸屬於母公司業主','基本每股盈餘（元）']}
    quarters={}
    for quarter,prior in [('2024Q2','2024Q1'),('2024Q3','2024Q2')]:
        v={k:selected[quarter][k]-selected[prior][k] for k in selected[quarter] if k in selected[prior] and k!='基本每股盈餘（元）'}
        revenue=v.get('營業收入')
        if revenue:
            v['gross_margin_pct']=v.get('營業毛利（毛損）',0)/revenue*100
            v['operating_margin_pct']=v.get('營業利益（損失）',0)/revenue*100
        quarters[quarter]=v
    rel=[a for a in d['announcements'] if any(k in (a['subject'] or '') for k in ['財務報','法說','法人'])]
    q3_ann=[a for a in rel if '第三季' in a['subject'] or '第3季' in a['subject']]
    revenues=[r for r in historical_a25_revenue_summary if r['year']==2024 and r['stock_id']==sid]
    check={'stock_id':sid,'cumulative_raw':selected,'quarterly_diff_excluding_EPS':quarters,'revenue_summary':revenues}
    second_batch_numeric_checks.append(check)
    print(json.dumps(check,ensure_ascii=False,default=str))
    print('Quarter-announcement originals:',json.dumps([{k:a[k] for k in ['announce_date','announce_time','subject','content']} for a in q3_ann],ensure_ascii=False,default=str))


In [ ]:
import urllib.request,re,ssl,certifi
try:
    req=urllib.request.Request('https://www.acter.com.tw/investors/conference',headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req,timeout=25,context=ssl.create_default_context(cafile=certifi.where())) as response: acter_conference_html=response.read().decode('utf-8')
    print('Official conference page script/API candidates:')
    for ln in acter_conference_html.splitlines():
        if any(k in ln for k in ['axios','ajax','getJSON','/api/','dataYear','url:','fetch(','new Vue']): print(ln.strip()[:500])
    print('Script paths:',re.findall(r'<script[^>]*src=["\']([^"\']+)',acter_conference_html)[-10:])
except Exception as source_error:
    print('Official dynamic page retrieval unavailable:',type(source_error).__name__,'; retain web sources and mark original-report retrieval incomplete.')


In [ ]:
import json,datetime
second_batch_comparison=[]
for check in second_batch_numeric_checks:
    q2=check['quarterly_diff_excluding_EPS']['2024Q2']
    q3=check['quarterly_diff_excluding_EPS']['2024Q3']
    result={'stock_id':check['stock_id']}
    for key in ['營業收入','營業利益（損失）','本期淨利（淨損）','淨利（淨損）歸屬於母公司業主']:
        result[key+'_qoq_pct']=(q3[key]/q2[key]-1)*100
    for name,q in [('Q2',q2),('Q3',q3)]:
        result[name+'_noncontrolling_net']=q['本期淨利（淨損）']-q['淨利（淨損）歸屬於母公司業主']
    decision=datetime.date(2024,11,8) if check['stock_id']=='3293' else datetime.date(2024,11,11)
    result['earliest_financial_announcement_decision']=str(decision)
    result['remaining_sessions_including_decision']=sum(decision<=d<=datetime.date(2024,12,2) for d in dates)
    second_batch_comparison.append(result)
print(json.dumps(second_batch_comparison,ensure_ascii=False,default=str))
assert all(x['remaining_sessions_including_decision'] in [16,17] for x in second_batch_comparison)
